# 04 — LangChain4j integration

`JavaVectorsEmbeddingStore` implements LangChain4j's `EmbeddingStore<TextSegment>`, so anywhere the framework expects an embedding store (`InMemoryEmbeddingStore`, Redis, pgvector…), `java-vectors` drops in without code changes.

This notebook builds a tiny retrieval loop: ingest a handful of `TextSegment`s, embed them with a deterministic model, query, and print matches with scores.

In [ ]:
%jars /home/jovyan/work/vectors/vectors-core/build/libs/vectors-core-*.jar
%jars /home/jovyan/work/vectors/vectors-storage/build/libs/vectors-storage-*.jar
%jars /home/jovyan/work/vectors/vectors-quantization/build/libs/vectors-quantization-*.jar
%jars /home/jovyan/work/vectors/vectors-hnsw/build/libs/vectors-hnsw-*.jar
%jars /home/jovyan/work/vectors/vectors-db/build/libs/vectors-db-*.jar
%jars /home/jovyan/work/vectors/vectors-langchain4j/build/libs/vectors-langchain4j-*.jar

%%loadFromPOM
<dependencies>
  <dependency>
    <groupId>dev.langchain4j</groupId>
    <artifactId>langchain4j-core</artifactId>
    <version>1.0.0-beta1</version>
  </dependency>
</dependencies>

In [ ]:
import com.integrallis.vectors.core.SimilarityFunction;
import com.integrallis.vectors.db.IndexType;
import com.integrallis.vectors.db.VectorCollection;
import com.integrallis.vectors.langchain4j.JavaVectorsEmbeddingStore;
import dev.langchain4j.data.embedding.Embedding;
import dev.langchain4j.data.segment.TextSegment;
import dev.langchain4j.model.embedding.EmbeddingModel;
import dev.langchain4j.model.output.Response;
import dev.langchain4j.store.embedding.EmbeddingMatch;
import dev.langchain4j.store.embedding.EmbeddingSearchRequest;
import dev.langchain4j.store.embedding.EmbeddingSearchResult;
import java.util.*;

/** Deterministic trigram embedder — good enough to show retrieval working. */
final class TrigramEmbedder implements EmbeddingModel {
    private final int dim;
    TrigramEmbedder(int dim) { this.dim = dim; }
    @Override public int dimension() { return dim; }
    @Override public Response<Embedding> embed(String text) { return Response.from(Embedding.from(trigram(text))); }
    @Override public Response<Embedding> embed(TextSegment s) { return embed(s.text()); }
    @Override public Response<List<Embedding>> embedAll(List<TextSegment> segs) {
        List<Embedding> out = new ArrayList<>(segs.size());
        for (TextSegment s : segs) out.add(Embedding.from(trigram(s.text())));
        return Response.from(out);
    }
    private float[] trigram(String text) {
        float[] v = new float[dim];
        String s = text.toLowerCase();
        if (s.length() < 3) s = "  " + s + "  ";
        for (int i = 0; i <= s.length() - 3; i++) {
            int h = 31*(31*s.charAt(i) + s.charAt(i+1)) + s.charAt(i+2);
            v[Math.floorMod(h, dim)] += 1f;
        }
        float n = 0f; for (float x : v) n += x*x; n = (float)Math.sqrt(n);
        if (n > 0) for (int i = 0; i < dim; i++) v[i] /= n;
        return v;
    }
}

In [ ]:
EmbeddingModel model = new TrigramEmbedder(128);

VectorCollection collection = VectorCollection.builder()
    .dimension(model.dimension())
    .metric(SimilarityFunction.COSINE)
    .indexType(IndexType.HNSW)
    .hnswM(16).hnswEfConstruction(100)
    .autoCommitThreshold(32)
    .build();

JavaVectorsEmbeddingStore store = JavaVectorsEmbeddingStore.builder(collection)
    .commitAfterAdd(true)
    .build();

List<TextSegment> segments = List.of(
    TextSegment.from("HNSW is a graph-based approximate nearest neighbor index."),
    TextSegment.from("Product quantization compresses vectors into short codes."),
    TextSegment.from("mmap maps a file into the process address space for zero-copy reads."),
    TextSegment.from("SIMD via Panama Vector API accelerates distance kernels on the JVM."),
    TextSegment.from("Scalar int8 quantization gives 4x memory reduction with <1% recall loss."));
List<Embedding> embeddings = model.embedAll(segments).content();
store.addAll(embeddings, segments);

Embedding qv = model.embed("What is SIMD on the JVM?").content();
EmbeddingSearchResult<TextSegment> result = store.search(
    EmbeddingSearchRequest.builder().queryEmbedding(qv).maxResults(3).build());
int i = 1;
for (EmbeddingMatch<TextSegment> m : result.matches()) {
    System.out.printf("  [%d] score=%.4f  %s%n", i++, m.score(), m.embedded().text());
}

collection.close();

## Next up

- **05 — Embedding cache:** wrap a `EmbeddingModel` like the one here with `CachingEmbeddingModel` and measure hit-rate across repeated queries.